# IMQCAM File Collections: Search, Inspect, and Download Data

This notebook shows modelers how to find IMQCAM data files using the IMQCAM-specific `/imqcam/datafiles` endpoint and download only the matching results.

`dataType` is now optional for `/imqcam/datafiles`. This means a modeler can start from the search criteria they know instead of being required to start from a data type. The endpoint itself only exposes items that have IGSN metadata.

The workflow supports:

- **IGSN**
- **Data type**
- **Composition**
- **File extension** (for example `.ang`, `.ctf`, `.csv`)
- combinations of those filters

### IGSN note

Some IMQCAM items store more than one IGSN in the same metadata field, for example:

`JHABOX00001-002, JHABOX00001-001`

To avoid missing those records, this notebook normalizes the stored IGSN values and performs constituent-IGSN matching after the API results are returned.

### Safe defaults

- All search filters may be left blank to retrieve all items exposed by `/imqcam/datafiles`.
- Downloads are **off by default**.
- Downloads use the same filtered results shown in the notebook.


## 1. Connect to the IMQCAM DMS

Run this notebook from the repository so that `imqcam.py` can be imported.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "imqcam.py").exists()
)
sys.path.insert(0, str(REPO_ROOT))

from imqcam import get_client
from imqcam_file_collections import download_items, fetch_datafiles

client = get_client()
print(f"Connected to {client.urlBase}")


## 2. Explore available data types

`GET /imqcam/datatype` lists the data types currently available in IMQCAM.


In [ ]:
data_types = client.get("imqcam/datatype")
print(data_types)


## 3. Count files

`GET /imqcam/count` returns file counts. With no parameters, it returns overall counts.


In [ ]:
counts = client.get(
    "imqcam/count",
    parameters={},
)
print(counts)


### Count files for one IGSN

The count endpoint can also be restricted to a specimen IGSN.


In [ ]:
example_igsn = "CMXMAG00001-042"

counts_for_igsn = client.get(
    "imqcam/count",
    parameters={"igsn": example_igsn},
)
print(counts_for_igsn)


## 4. Retrieve data files

`GET /imqcam/datafiles` is the main endpoint used below.

`dataType` is optional. When it is omitted, the endpoint can return all accessible data files, subject to any metadata filters supplied in the request.


In [ ]:
example_items = client.get(
    "imqcam/datafiles",
    parameters={
        "dataType": "ebsd_test",  # optional
        "limit": 5,
    },
)
print(f"Returned {len(example_items)} item(s)")


## 5. Choose search filters

`/imqcam/datafiles` only exposes items that have IGSN metadata. Providing an IGSN as a **search argument** is optional.

Set any filters you know:

- `IGSN`: optional; one IGSN, comma-separated IGSNs, or a Python list
- `DATA_TYPE`: optional; for example `ebsd_raw`
- `COMPOSITION_FILTER`: optional; for example `IN718`
- `FILE_EXTENSION`: optional; for example `.ang`, `.ctf`, or `.csv`

Leave a filter blank to ignore it.

If **all search filters are blank**, the notebook retrieves items exposed by `/imqcam/datafiles`. Items without IGSN metadata are excluded by the endpoint itself.

The search follows this pattern:

`optional API criteria -> /imqcam/datafiles -> local IGSN/composition/extension checks -> filtered results`

`dataType` is sent to `/imqcam/datafiles` only when provided.

Composition is requested from `/imqcam/datafiles` through:

`extraFields=["meta.composition"]`

and then checked locally against each returned item's metadata.

> Note: macOS AppleDouble sidecar files whose names begin with `._` are ignored by the file-extension filter.


In [ ]:
# IGSN is an optional SEARCH filter.
# /imqcam/datafiles itself only exposes items that already have IGSN metadata.
IGSN = ""
DATA_TYPE = ""
COMPOSITION_FILTER = ""
FILE_EXTENSION = ""  # e.g. ".ang", ".ctf", ".csv"

PAGE_SIZE = 100
MAX_RESULTS = None  # Use None to fetch every matching item exposed by the endpoint


## 6. Search helpers

These helpers:

1. normalize one or many requested IGSNs;
2. call `/imqcam/datafiles` with optional `dataType`;
3. request `meta.composition` through `extraFields`;
4. paginate through the endpoint results;
5. perform constituent-IGSN matching locally so comma-separated IGSN metadata is handled correctly;
6. apply composition and file-extension filtering locally.

`meta.composition` cannot be used directly in the endpoint's `filters` parameter, so it is requested through `extraFields` and then filtered locally.


In [ ]:
# Search and download helpers are kept in imqcam_file_collections.py.
# They validate pagination and preserve the search settings with the results.


## 7. Run the search

`/imqcam/datafiles` performs the API query first. The endpoint only exposes items that have IGSN metadata.

The notebook requests `meta.composition` using `extraFields`, then applies any IGSN, composition, or extension checks locally. Extension filtering checks returned **item names** only; it does not inspect a folder item's contents.

The returned `search_results` retains the exact normalized settings used for this search. The same matching items and saved settings are used later for display and download.


In [ ]:
def report_search_progress(page_number, fetched, matched):
    print(
        f"Search progress: page {page_number:,}; "
        f"fetched {fetched:,} item(s); matched {matched:,}."
    )


search_results = fetch_datafiles(
    client=client,
    igsn=IGSN,
    data_type=DATA_TYPE,
    composition=COMPOSITION_FILTER,
    file_extension=FILE_EXTENSION,
    page_size=PAGE_SIZE,
    max_results=MAX_RESULTS,
    progress_callback=report_search_progress,
)
items = search_results.items
settings = search_results.settings

selected_filters = []

if settings.igsns:
    selected_filters.append(f"IGSN={', '.join(settings.igsns)}")

if settings.data_type:
    selected_filters.append(f"data type={settings.data_type}")

if settings.composition:
    selected_filters.append(f"composition={settings.composition}")

if settings.file_extension:
    selected_filters.append(f"file extension={settings.file_extension}")

if selected_filters:
    print(
        f"Matched {len(items):,} item(s) for selected filters: "
        + ", ".join(selected_filters)
    )
else:
    print(
        f"Matched {len(items):,} item(s) with no search filters "
        "(items exposed by /imqcam/datafiles)."
    )


## 8. Inspect matching metadata

Review the matches before downloading. The table below shows the key metadata fields for each returned item.


In [ ]:
rows = []

for item in items:
    meta = item.get("meta", {}) or {}

    rows.append(
        {
            "name": item.get("name"),
            "item_id": item.get("_id"),
            "igsn": meta.get("igsn"),
            "composition": meta.get("composition"),
            "data_type": meta.get("data_type"),
            "extension": Path(str(item.get("name") or "")).suffix.lower(),
        }
    )

results_df = pd.DataFrame(
    rows,
    columns=["name", "item_id", "igsn", "composition", "data_type", "extension"],
)

display(results_df.head(20))


## 9. Download the matched items

Downloads are disabled by default. Review `results_df` first. When the results are correct, set:

```python
DOWNLOAD_FILES = True
```

Only the items and search settings returned by the active query are used. Changing filter variables after a search does not change the download destination.

`DOWNLOAD_LIMIT` is set to `5` by default as an additional safeguard. Set it to `None` only when you intend to download every matched item. An extension filter selects by item name, and a matching item is downloaded in full.


In [ ]:
DOWNLOAD_FILES = False
DOWNLOAD_LIMIT = 5  # Set to None to download every matched item


if DOWNLOAD_FILES:
    download_items(
        client,
        search_results,
        root=REPO_ROOT,
        limit=DOWNLOAD_LIMIT,
    )
else:
    print(
        "Downloads are OFF. Review the matched items first, then set "
        "DOWNLOAD_FILES = True when ready."
    )


## Summary

For most modeler searches, use Sections 5–9:

1. choose any search filters you know;
2. query `/imqcam/datafiles` with optional `dataType`;
3. request `meta.composition` through `extraFields`;
4. apply constituent-IGSN matching locally when an IGSN is supplied;
5. apply composition and item-name extension checks locally;
6. inspect the returned metadata;
7. enable downloads only after confirming the matches.

`dataType` is optional. Providing an IGSN as a search argument is also optional, but `/imqcam/datafiles` itself only exposes items that already have IGSN metadata.

The saved search settings determine the filter-specific download directory. An extension filter checks an item's name, not the contents of a folder item; matching items are downloaded in full.
